In [2]:
# DimDate: generated in PySpark rather than Gold SQL, since Synapse SQL
# doesn't support recursive CTEs (the standard way to generate a date series).
# DayOfMonth is computed here too, not as a Direct Lake calculated column —
# Direct Lake semantic models only support DAX measures, not calculated
# columns, so sortable/groupable fields like this must be pushed upstream
# into the source layer.
# Written to the Gold Warehouse via the synapsesql connector.

from pyspark.sql import functions as F
import com.microsoft.spark.fabric
dim_date = (
    spark.range(1)
    .select(
        F.explode(
            F.sequence(
                F.to_date(F.expr("'2016-01-01'")),
                F.to_date(F.expr("'2018-12-31'"))
            )
        ).alias("Date")
    )
    .withColumn("DateKey", F.date_format("Date", "yyyyMMdd").cast("int"))
    .withColumn("Year", F.year("Date"))
    .withColumn("Month", F.month("Date"))
    .withColumn("MonthName", F.date_format("Date", "MMMM"))
    .withColumn("Quarter", F.quarter("Date"))
    .withColumn("WeekdayName", F.date_format("Date", "EEEE"))
    .withColumn("IsWeekend", F.when(F.dayofweek("Date").isin([1, 7]), 1).otherwise(0))
    .withColumn("DayOfMonth", F.dayofmonth("Date"))
)

dim_date.write.mode("overwrite").synapsesql("wh_gold_olist.dbo.DimDate")

StatementMeta(, 05897cef-4d36-43bb-bc3d-4787487c3086, 4, Finished, Available, Finished, False)